In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# =============================================================================
# 1. MARKOV REGIME MODEL (EXACTLY AS PROVIDED)
# =============================================================================
class MarkovRegime:
    def __init__(self):
        self.n_states = 3
        self.current_state = 0 
        self.colors = ['#3fb950', '#d29922', '#f85149'] 
        self.bg_colors = ['rgba(63, 185, 80, 0.25)', 'rgba(210, 153, 34, 0.25)', 'rgba(248, 81, 73, 0.25)']
        self.state_probs = np.array([1/3, 1/3, 1/3])
        self.transition_matrix = np.array([
            [0.90, 0.08, 0.02],  
            [0.10, 0.80, 0.10],  
            [0.02, 0.08, 0.90]   
        ])
        self.emission_means = np.array([0.0005, 0.002, 0.005])
        self.emission_stds = np.array([0.0003, 0.001, 0.003])

    def calibrate(self, hist_bars):
        if len(hist_bars) < 20: return
        vols = np.array([(b['h'] - b['l']) / b['c'] if b['c'] > 0 else 0 for b in hist_bars])
        vols = vols[vols > 0]
        if len(vols) < 20: return

        p33, p67 = np.percentile(vols, 33), np.percentile(vols, 67)
        regime_assignments = np.zeros(len(vols), dtype=int)
        regime_assignments[vols >= p33] = 1
        regime_assignments[vols >= p67] = 2

        for regime in range(self.n_states):
            regime_vols = vols[regime_assignments == regime]
            if len(regime_vols) >= 3:
                self.emission_means[regime] = np.mean(regime_vols)
                self.emission_stds[regime] = max(np.std(regime_vols), 1e-6)

        sorted_indices = np.argsort(self.emission_means)
        self.emission_means = self.emission_means[sorted_indices]
        self.emission_stds = self.emission_stds[sorted_indices]

        transition_counts = np.zeros((self.n_states, self.n_states))
        for t in range(1, len(regime_assignments)):
            prev_regime = regime_assignments[t-1]
            curr_regime = regime_assignments[t]
            transition_counts[prev_regime, curr_regime] += 1

        for i in range(self.n_states):
            row_sum = transition_counts[i].sum()
            if row_sum > 0:
                self.transition_matrix[i] = (transition_counts[i] + 0.1) / (row_sum + 0.3)
        self.state_probs = np.array([1/3, 1/3, 1/3])

    def _gaussian_likelihood(self, vol, regime):
        mean = self.emission_means[regime]
        std = self.emission_stds[regime]
        coeff = 1 / (std * np.sqrt(2 * np.pi))
        exponent = -0.5 * ((vol - mean) / std) ** 2
        return coeff * np.exp(exponent)

    def get_regime(self, bars):
        if not bars: return self.current_state
        current_bar = bars[-1]
        vol = current_bar.volatility
        if vol <= 0:
            current_bar.regime = self.current_state
            return self.current_state

        prior_probs = self.transition_matrix.T @ self.state_probs
        likelihoods = np.array([self._gaussian_likelihood(vol, i) for i in range(self.n_states)])
        posterior_probs = prior_probs * likelihoods

        prob_sum = posterior_probs.sum()
        if prob_sum > 0:
            posterior_probs = posterior_probs / prob_sum
        else:
            posterior_probs = prior_probs

        self.state_probs = posterior_probs
        self.current_state = int(np.argmax(posterior_probs))
        current_bar.regime = self.current_state
        return self.current_state

# =============================================================================
# 2. BAR CLASS (EXACTLY AS PROVIDED)
# =============================================================================
class Bar:
    def __init__(self, datetime, open_price, high, low, close, volume):
        self.datetime = datetime
        self.o = open_price
        self.h = high
        self.l = low
        self.c = close
        self.v = volume
        self.volatility = (high - low) / close if close > 0 else 0
        self.regime = 0

# =============================================================================
# 3. CSV PROCESSING LOGIC
# =============================================================================
def label_csv_with_regimes(input_csv, output_csv):
    """
    Reads your engineered CSV, runs it through the custom Markov logic,
    and saves a new CSV with the Regime labels attached.
    """
    print(f"Loading data from {input_csv}...")
    # Load the CSV. Assumes the Date is the first column (index)
    df = pd.read_csv(input_csv, index_col=0, parse_dates=True)
    
    # Create the Bar objects row by row
    bars = []
    for date_index, row in df.iterrows():
        bar = Bar(
            datetime=date_index,
            open_price=row['XAUUSD_Open'],
            high=row['XAUUSD_High'],
            low=row['XAUUSD_Low'],
            close=row['XAUUSD_Close'],
            volume=0  # Volume isn't in your CSV snippet, and isn't used in the math
        )
        bars.append(bar)

    # Initialize and Calibrate the Model
    print("Calibrating Markov Model...")
    regime_model = MarkovRegime()
    calibration_size = min(100, len(bars))
    regime_model.calibrate([{'h': b.h, 'l': b.l, 'c': b.c} for b in bars[:calibration_size]])

    # Process all bars to find their regime
    print("Labeling regimes across dataset...")
    regimes = []
    for i in range(len(bars)):
        current_bars = bars[:i+1]
        regime = regime_model.get_regime(current_bars)
        regimes.append(regime)

    # Add the results back to the dataframe
    df['Regime'] = regimes
    
    # Save to the new CSV
    df.to_csv(output_csv, index=True)
    print(f"✅ Finished! Data with Regime labels saved to {output_csv}")

# =============================================================================
# EXECUTE
# =============================================================================
if __name__ == "__main__":
    BASE_DIR = Path.cwd()
    DATABASE_DIR = BASE_DIR / 'database'
    INPUT_FILE = DATABASE_DIR / 'feature_matrix.csv'
    OUTPUT_FILE = DATABASE_DIR / 'labeled_feature_matrix.csv'
    
    label_csv_with_regimes(INPUT_FILE, OUTPUT_FILE)

Loading data from database\feature_matrix.csv...
Calibrating Markov Model...
Labeling regimes across dataset...
✅ Finished! Data with Regime labels saved to database\labeled_feature_matrix.csv
